# 02 Feature Engineering

Build chronological, leakage-safe team features from prior matches only.

In [ ]:
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

def resolve_artifacts_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd / "artifacts", cwd.parent / "artifacts"]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return cwd.parent / "artifacts" if cwd.name == "notebooks" else cwd / "artifacts"


ARTIFACTS_DIR = resolve_artifacts_dir()


def expected_score(rating_a: float, rating_b: float) -> float:
    return 1.0 / (1.0 + 10 ** ((rating_b - rating_a) / 400.0))


def safe_mean(values: list[int], default: float = 0.5) -> float:
    return float(np.mean(values)) if values else default


def build_match_features_local(
    df: pd.DataFrame,
    base_elo: float = 1500.0,
    k_factor: float = 32.0,
) -> pd.DataFrame:
    ordered = df.sort_values(["date"], kind="mergesort").reset_index(drop=True).copy()
    elo = defaultdict(lambda: base_elo)
    matches_played = defaultdict(int)
    overall_history = defaultdict(list)
    blue_side_history = defaultdict(list)
    red_side_history = defaultdict(list)
    last_played = {}
    head_to_head = defaultdict(list)
    rows = []

    for row in ordered.itertuples(index=False):
        blue = row.blue_team
        red = row.red_team
        match_date = row.date
        key = tuple(sorted((blue, red)))
        blue_h2h = head_to_head[key]

        rows.append(
            {
                "season": row.season,
                "date": match_date,
                "event": row.event,
                "patch": row.patch,
                "blue_team": blue,
                "red_team": red,
                "winner": row.winner,
                "blue_team_win": row.blue_team_win,
                "elo_diff": elo[blue] - elo[red],
                "winrate_last_5_diff": safe_mean(overall_history[blue][-5:]) - safe_mean(overall_history[red][-5:]),
                "winrate_last_10_diff": safe_mean(overall_history[blue][-10:]) - safe_mean(overall_history[red][-10:]),
                "winrate_last_20_diff": safe_mean(overall_history[blue][-20:]) - safe_mean(overall_history[red][-20:]),
                "matches_played_diff": matches_played[blue] - matches_played[red],
                "days_since_last_match_diff": (
                    (match_date - last_played[blue]).days if blue in last_played else -1
                ) - (
                    (match_date - last_played[red]).days if red in last_played else -1
                ),
                "head_to_head_winrate_diff": (
                    sum(1 for winner in blue_h2h if winner == blue) / len(blue_h2h) if blue_h2h else 0.5
                ) - 0.5,
                "blue_side_team_winrate": safe_mean(blue_side_history[blue]),
                "red_side_team_winrate": safe_mean(red_side_history[red]),
            }
        )

        if pd.isna(row.blue_team_win):
            continue

        outcome = int(row.blue_team_win)
        expected = expected_score(elo[blue], elo[red])
        elo[blue] += k_factor * (outcome - expected)
        elo[red] += k_factor * ((1 - outcome) - (1 - expected))
        matches_played[blue] += 1
        matches_played[red] += 1
        overall_history[blue].append(outcome)
        overall_history[red].append(1 - outcome)
        blue_side_history[blue].append(outcome)
        red_side_history[red].append(1 - outcome)
        last_played[blue] = match_date
        last_played[red] = match_date
        head_to_head[key].append(blue if outcome == 1 else red)

    return pd.DataFrame(rows)


In [ ]:
clean_matches = pd.read_csv(ARTIFACTS_DIR / "clean_matches.csv", parse_dates=["date"])
match_features = build_match_features_local(clean_matches)
match_features.to_csv(ARTIFACTS_DIR / "match_features.csv", index=False)
feature_df = match_features
feature_df.head()


In [ ]:
feature_columns = [
    "elo_diff",
    "winrate_last_5_diff",
    "winrate_last_10_diff",
    "winrate_last_20_diff",
    "matches_played_diff",
    "days_since_last_match_diff",
    "head_to_head_winrate_diff",
    "blue_side_team_winrate",
    "red_side_team_winrate",
]
feature_df[feature_columns].describe().T

In [ ]:
feature_df[["date", "blue_team", "red_team", "blue_team_win", "elo_diff"]].head(10)

In [ ]:
feature_df.isna().mean().sort_values(ascending=False).head(20)